In [0]:
# =========================================================================
# GOLD LAYER: Master Orchestrator (Driver Notebook)
# =========================================================================
# Description: Sequentially executes all Gold layer worker notebooks 
# to build and register the Star Schema tables in Unity Catalog.
# =========================================================================

import time

print("🚀 Starting Gold Layer Pipeline Orchestration...")
pipeline_start_time = time.time()

# Define the sequence of worker notebooks to execute.
# Ensure these notebook files are saved in the same workspace folder.
gold_pipeline_steps = [
    ("Gold Dimension: Customers", "dim_customers"),
    ("Gold Dimension: Products", "dim_products"),
    ("Gold Dimension: Calendar", "dim_calendar"),
    ("Gold Dimension: Territories", "dim_territories"),
    ("Gold Fact: Sales", "fact_sales")
]

# Execute each worker notebook sequentially
for step_name, notebook_path in gold_pipeline_steps:
    print(f"\n" + "="*60)
    print(f"▶️ Triggering: {step_name}")
    print(f"📂 Notebook: {notebook_path}")
    print("="*60)
    
    step_start_time = time.time()
    
    try:
        # dbutils.notebook.run executes the worker notebook in an isolated context 
        # and waits for it to complete (Timeout set to 10 minutes / 600 seconds)
        output = dbutils.notebook.run(notebook_path, timeout_seconds=600)
        
        step_duration = time.time() - step_start_time
        print(f"✅ Successfully completed: {step_name}")
        print(f"⏱️ Duration: {step_duration:.2f} seconds")
        print(f"📌 Worker Output: {output}")
        
    except Exception as e:
        print(f"\n❌ PIPELINE FAILURE at step: {step_name}")
        print(f"🔍 Error Details: {str(e)}")
        # Raise error to halt execution immediately on failure
        raise e

# Pipeline Summary
total_duration = time.time() - pipeline_start_time
print(f"\n" + "="*60)
print(f"🎉 GOLD LAYER PIPELINE COMPLETED SUCCESSFULLY!")
print(f"⏱️ Total Execution Time: {total_duration:.2f} seconds")
print(f"🏛️ All tables are updated in storage and registered under Unity Catalog (`gold.*`).")
print("="*60)